In [ ]:

def _get_target_plane_uv_bounds(self) -> tuple:
        """Get UV bounds of target plane in world space."""
        plane = self.target_plane_obj
        
        # Get plane vertices
        bm = bmesh.new()
        bm.from_mesh(plane.data)
        
        # Alternative: use plane mesh vertices directly
        plane_mesh = plane.data
        vertices_world = [plane.matrix_world @ v.co for v in plane_mesh.vertices]
        
        vertices_world = np.array([list(v) for v in vertices_world])
        
        bm.free()
        
        return vertices_world
    
    def _world_to_plane_uv(self, world_point: Vector) -> tuple:
        """Convert a world point to UV coordinates on target plane."""
        plane = self.target_plane_obj
        plane_mesh = plane.data
        
        # Get plane vertices (assuming a simple rectangular plane)
        verts = [plane.matrix_world @ v.co for v in plane_mesh.vertices]
        
        if len(verts) < 4:
            raise ValueError("Target plane must have at least 4 vertices")
        
        # For a simple rectangular plane, use first 4 verts
        v0 = verts[0]
        v1 = verts[1]
        v2 = verts[2]
        v3 = verts[3]
        
        # Create plane basis vectors
        edge1 = v1 - v0
        edge2 = v3 - v0
        
        # Project world point onto plane
        to_point = world_point - v0
        
        u = to_point.dot(edge1) / edge1.dot(edge1)
        v = to_point.dot(edge2) / edge2.dot(edge2)
        
        return (u, v)
    
    def _shoot_ray(self, screen_x: float, screen_y: float) -> Vector:
        """
        Shoot a ray from camera through a screen coordinate.
        
        Args:
            screen_x: X coordinate in mask (0 to mask_width)
            screen_y: Y coordinate in mask (0 to mask_height)
        
        Returns:
            3D point where ray intersects target plane, or None if no intersection
        """
        # Normalize screen coordinates to [-1, 1]
        # Need to use render dimensions, not mask dimensions
        render = self.scene.render
        render_x = (screen_x / self.mask_width) * render.resolution_x
        render_y = (screen_y / self.mask_height) * render.resolution_y
        
        # Create ray from camera
        camera = self.camera_obj.data
        
        # Get camera center in world space
        camera_pos = self.camera_obj.matrix_world.translation
        
        # Calculate ray direction using camera intrinsics
        # Normalize to camera space
        cam_x = (render_x - self.principal_point_x) / self.focal_length_px
        cam_y = (render_y - self.principal_point_y) / self.focal_length_px
        
        # Ray direction in camera space
        ray_cam = Vector((cam_x, cam_y, -1)).normalized()
        
        # Transform ray to world space
        rot = self.camera_obj.matrix_world.to_3x3()
        ray_world = rot @ ray_cam
        
        # Intersect with target plane
        intersection = self._ray_plane_intersection(camera_pos, ray_world)
        
        return intersection
    
    def _ray_plane_intersection(self, ray_origin: Vector, ray_dir: Vector) -> Vector:
        """
        Find intersection of ray with target plane.
        
        Args:
            ray_origin: Starting point of ray
            ray_dir: Direction of ray
        
        Returns:
            3D intersection point or None
        """
        plane = self.target_plane_obj
        plane_mesh = plane.data
        
        # Get plane normal and point
        verts = [plane.matrix_world @ v.co for v in plane_mesh.vertices]
        
        v0 = verts[0]
        v1 = verts[1]
        v2 = verts[2]
        
        # Calculate plane normal
        edge1 = v1 - v0
        edge2 = v2 - v0
        plane_normal = edge1.cross(edge2).normalized()
        
        # Ray-plane intersection
        denom = ray_dir.dot(plane_normal)
        
        if abs(denom) < 1e-6:
            return None  # Ray is parallel to plane
        
        t = (v0 - ray_origin).dot(plane_normal) / denom
        
        if t < 0:
            return None  # Intersection behind ray origin
        
        intersection = ray_origin + t * ray_dir
        
        return intersection
    
    def project_mask(self) -> np.ndarray:
        """
        Project mask onto scene and create output texture.
        
        Returns:
            Output texture as numpy array (H, W, 4)
        """
        mask_array = np.array(self.mask_image)
        
        # Get UV bounds of output texture
        plane_verts = self._get_target_plane_uv_bounds()
        
        # Process each pixel in mask
        for y in range(self.mask_height):
            for x in range(self.mask_width):
                alpha = mask_array[y, x, 3]
                
                # Skip transparent pixels
                if alpha < 200:
                    continue
                
                # Shoot ray and get intersection
                intersection_point = self._shoot_ray(x, y)
                
                if intersection_point is None:
                    continue
                
                # Convert world point to plane UV
                u, v = self._world_to_plane_uv(intersection_point)
                
                # Clamp to [0, 1]
                u = max(0, min(1, u))
                v = max(0, min(1, v))
                
                # Convert to texture coordinates
                tex_x = int(u * (self.texture_resolution[0] - 1))
                tex_y = int(v * (self.texture_resolution[1] - 1))
                
                # Clamp to texture bounds
                tex_x = max(0, min(self.texture_resolution[0] - 1, tex_x))
                tex_y = max(0, min(self.texture_resolution[1] - 1, tex_y))
                
                # Copy pixel from mask to output texture
                self.output_texture[tex_y, tex_x] = mask_array[y, x]
        
        return self.output_texture
    
    def save_texture(self, output_path: str = None):
        """
        Save projected texture to file.
        
        Args:
            output_path: Path to save texture (defaults to self.output_path)
        """
        output_path = output_path or self.output_path
        
        # Convert to PIL Image and save
        output_img = Image.fromarray(self.output_texture, mode='RGBA')
        output_img.save(output_path)
        
        print(f"Saved projected texture to {output_path}")
        return output_path


def project_mask(
    mask_path: str,
    camera_obj: bpy.types.Object = None,
    target_plane_obj: bpy.types.Object = None,
    output_path: str = None,
    texture_resolution: tuple = (2048, 2048),
) -> str:
    """
    Main function to project a mask onto a scene.
    
    Args:
        mask_path: Path to input PNG mask image
        camera_obj: Blender camera object (defaults to scene camera)
        target_plane_obj: Blender plane object to project onto
        output_path: Path to save output texture
        texture_resolution: Output texture resolution
    
    Returns:
        Path to saved output texture
    
    Example:
        >>> project_mask(
        ...     mask_path="/path/to/mask.png",
        ...     target_plane_obj=bpy.data.objects['Plane'],
        ...     output_path="/path/to/output.png"
        ... )
    """
    projector = MaskProjector(
        mask_path=mask_path,
        camera_obj=camera_obj,
        target_plane_obj=target_plane_obj,
        output_path=output_path,
        texture_resolution=texture_resolution,
    )
    
    # Project mask
    projector.project_mask()
    
    # Save result
    return projector.save_texture()


In [1]:
import bpy
import numpy as np
from PIL import Image
from mathutils import Vector, Matrix, Quaternion
import bmesh
from bpy_extras.view3d_utils import region_2d_to_origin_3d, region_2d_to_vector_3d
import os
import bruteForce as bf
import json
import math

c:\Users\dario\work\find_camera_params\RDD\models\ops\functions\ms_deform_attn_func.py:31: UserWarning: 
The MultiScaleDeformableAttention CUDA extension is not compiled. Using the slower PyTorch implementation instead.
	To compile it, run the following commands:
	cd RDD/models/ops
	pip install -e .
  warnings.warn(info_string)
c:\Users\dario\work\find_camera_params\RDD\matchers\lightglue.py:29: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


config {'activation': 'relu', 'block_dims': [8, 16, 32, 64], 'd_model': 256, 'detection_threshold': 0.1, 'device': 'cuda', 'dim_feedforward': 1024, 'dropout': 0.1, 'enc_n_points': 8, 'hidden_dim': 256, 'lr_backbone': 2e-05, 'nhead': 8, 'num_encoder_layers': 4, 'num_feature_levels': 5, 'top_k': 4096, 'train_detector': False, 'weights': './weights/RDD-v2.pth'}


c:\Users\dario\work\find_camera_params\RDD\RDD.py:260: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(config['weights'], map_location='cpu'))

In [2]:
render_settings = json.load(open(os.path.join("./render_settings.json"), "r"))
#set scene
bf.set_scene(render_settings=render_settings)
#set camera parameters

# it_height_1.214285945892334_z_rot_1.4361566416410483_x_rot_1.5707963267948966
# x_rot almost 90 degrees, z_rot around 82 degrees, height around 1.2142. This means the camera is looking at the horizon (x_rot 90)
x0 = [0.0, 0.0, 1.388, 1.0471, 0.0, 3.2313, math.radians(70)]
bf.set_camera_from_params(x0)
#test render

ROOT_PATH = "C:/Users/dario/work/find_camera_params/"
LOCATIONS_PATH = "C:/Users/dario/work/find_camera_params/test.png"

img_p = os.path.join(LOCATIONS_PATH)

bf.render_to_image(img_p,(int(1280/4), int(960/4)))
img_p = os.path.join(ROOT_PATH, "mask.png")
mask_image = Image.open(img_p).convert("RGBA")
mask_array = np.array(mask_image)
mask_h,mask_w = mask_array.shape[:2]
print(mask_h,mask_w)
cam = bpy.data.objects['Camera']
frame = cam.data.view_frame(scene=bpy.context.scene)#
print(frame)


removing mesh <bpy_struct, Mesh("Cube") at 0x0000013799020508>
960 1280
(Vector((0.5, 0.375, -0.7140740156173706)), Vector((0.5, -0.375, -0.7140740156173706)), Vector((-0.5, -0.375, -0.7140740156173706)), Vector((-0.5, 0.375, -0.7140740156173706)))


82.28571428571429

In [ ]:
def shoot_ray(x, y) -> float:
    
    render_res_x = bpy.context.scene.render.resolution_x
    render_res_y = bpy.context.scene.render.resolution_y

    #render and mask should have same dimensions
    #render_x = (screen_x / self.mask_width) * render_res_x
    #render_y = (screen_y / self.mask_height) * render_res_y
    
    camera = bpy.data.objects.get('Camera')
        
        # Create ray from camera
        
        # Get camera center in world space
    camera_pos = camera.matrix_world.translation
        
        # Calculate ray direction using camera intrinsics
        # Normalize to camera space
    focal_length = camera.data.lens
    cam_x = (x - self.principal_point_x) / focal_length
    cam_y = (y - self.principal_point_y) / focal_length
        
    # Ray direction in camera space
    ray_cam = Vector((cam_x, cam_y, -1)).normalized()
    
    # Transform ray to world space
    rot = camera.matrix_world.to_3x3()
    ray_world = rot @ ray_cam
        
        
        # Get evaluated depsgraph
    depsgraph = bpy.context.evaluated_depsgraph_get()
    
    # Perform ray cast
    result, location, normal, index, obj, matrix = bpy.context.scene.ray_cast(
        depsgraph,
        camera_pos,
        ray_world
    )

    if result:
        print("Hit object:", obj.name)
        print("Location:", location)
        print("Normal:", normal)
        return location[2]
    else:
        print("No hit")
        return 0.1

In [4]:
import bpy
from mathutils import Vector, Quaternion
import numpy as np
import bmesh

# objects to consider

# camera object which defines ray source
cam = bpy.data.objects['Camera']

# save current view mode
# mode = bpy.context.area.type

# set view mode to 3D to have all needed variables available
# bpy.context.area.type = "VIEW_3D"

# get vectors which define view frustum of camera
frame = cam.data.view_frame(scene=bpy.context.scene)

# frame2 = [cam.matrix_world @ corner for corner in frame]
topRight = frame[0]
bottomRight = frame[1]
bottomLeft = frame[2]
topLeft = frame[3]

# number of pixels in X/Y direction
resolutionX = 100 # int(bpy.context.scene.render.resolution_x * (bpy.context.scene.render.resolution_percentage / 100))
resolutionY = 100 # int(bpy.context.scene.render.resolution_y * (bpy.context.scene.render.resolution_percentage / 100))

# setup vectors to match pixels
xRange = np.linspace(topLeft[0], topRight[0], resolutionX)
yRange = np.linspace(topLeft[1], bottomLeft[1], resolutionY)

# array to store hit information
values = np.empty((xRange.size, yRange.size), dtype=object)

# indices for array mapping
indexX = 0
indexY = 0

# filling array with None
for x in xRange:
    for y in yRange:
        values[indexX,indexY] = None
        indexY += 1
    indexX += 1
    indexY = 0

# iterate over all targets
# calculate origin
matrixWorld = cam.matrix_world.to_3x3()
# matrixWorld = target.matrix_world
matrixWorldInverted = matrixWorld.inverted()
origin = cam.matrix_world.translation
# origin = matrixWorldInverted @ cam.matrix_world.translation
print(origin)            
    # reset indices
indexX = 0
indexY = 0
depsgraph = bpy.context.evaluated_depsgraph_get()

# iterate over all X/Y coordinates
for x in xRange:
    for y in yRange:
            # get current pixel vector from camera center to pixel
        pixelVector = Vector((x, y, topLeft[2]))
        destination = cam.matrix_world @ pixelVector
        direction = (destination - origin).normalized()    
    # Perform ray cast
        result, location, normal, index, obj, matrix = bpy.context.scene.ray_cast(
        depsgraph,
        origin,
        direction
    )


        if result:
            values[indexX,indexY] = (location)
            # values[indexX,indexY] = (matrixWorld @ location)
            
            # update indices
        indexY += 1
                        
    indexX += 1
    indexY = 0

<Vector (0.0000, 0.0000, 1.2142)>


In [5]:
x = xRange[52]
y = yRange[76]
pixelVector = Vector((x, y, topLeft[2]))
destination = cam.matrix_world @ pixelVector
direction = (destination - origin).normalized()
result, location, normal, face_index, obj, matrix = bpy.context.scene.ray_cast(
        depsgraph,
        origin,
        direction
    )
print(location)
print(obj.name)

<Vector (-0.8341, 0.1149, 1.2004)>
Plane


In [25]:
from mathutils.geometry import closest_point_on_tri
from mathutils.geometry import barycentric_transform
def get_uv_from_world_point(obj, world_point, face_index):
    """Get UV coordinates for a point on mesh surface."""
    # Get evaluated mesh (includes modifiers)
    depsgraph = bpy.context.evaluated_depsgraph_get()
    obj_eval = obj.evaluated_get(depsgraph)
    mesh = obj_eval.data
    
    bm = bmesh.new()
    bm.from_mesh(mesh)
    
    # Get UV layer
    uv_layer = bm.loops.layers.uv.active
    if uv_layer is None:
        raise ValueError("Mesh has no UV map")
    
    # Convert world point to local space of evaluated object
    local_point = obj_eval.matrix_world.inverted() @ world_point
    
    bm.faces.ensure_lookup_table()
    face = bm.faces[face_index]
    
    if len(face.verts) < 3:
        bm.free()
        return None
    
    # Use first 3 verts for triangle
    v0 = face.verts[0].co
    v1 = face.verts[1].co
    v2 = face.verts[2].co
    
    # Calculate barycentric coordinates
    bary = barycentric_transform(local_point, v0, v1, v2, Vector((1,0,0)), Vector((0,1,0)), Vector((0,0,1)))
    u, v, w = bary
    
    # Get UV coordinates from face loops
    uv0 = face.loops[0][uv_layer].uv
    uv1 = face.loops[1][uv_layer].uv
    uv2 = face.loops[2][uv_layer].uv
    
    # Interpolate UV using barycentric coordinates
    uv_result = u * uv0 + v * uv1 + w * uv2
    print(uv0,uv1,uv2)
    print(u, v, w)
    print(local_point)
    print(v0,v1,v2)
    bm.free()
    return uv_result

uv1 = get_uv_from_world_point(obj, location,face_index)
print(uv1)


<Vector (0.2875, 0.5250)> <Vector (0.2938, 0.5250)> <Vector (0.2938, 0.5312)>
0.36461278796195984 0.04088004678487778 0.5945071578025818
<Vector (-0.8341, 0.1149, 1.2004)>
<Vector (-0.8500, 0.1000, 1.2013)> <Vector (-0.8250, 0.1000, 1.1970)> <Vector (-0.8250, 0.1250, 1.2000)>
<Vector (0.2915, 0.5287)>


In [128]:
# Extract Z values from the values array, replacing None with 0
z_values = np.zeros((resolutionX, resolutionY))

for i in range(resolutionX):
    for j in range(resolutionY):
        if values[i, j] is not None:
            z_values[i, j] = np.linalg.norm(values[i, j] - origin)
        else:
            z_values[i, j] = 0

# Normalize to 0-255 range for image
z_min = np.nanmin(z_values[z_values > 0])
z_max = np.nanmax(z_values)
z_normalized = ((z_values - z_min) / (z_max - z_min) * 255).astype(np.uint8)

# Convert to PIL Image and save
img = Image.fromarray(z_normalized.transpose(), mode='L')
img.save(os.path.join(ROOT_PATH, 'values_image.png'))
print("Saved values as image to", os.path.join(ROOT_PATH, 'values_image.png'))

Saved values as image to C:/Users/dario/work/find_camera_params/values_image.png


In [ ]:

def world_to_plane_uv(self, world_point: Vector) -> tuple:
    """Convert a world point to UV coordinates on target plane."""
    plane = self.target_plane_obj
    plane_mesh = plane.data
    
    # Get plane vertices (assuming a simple rectangular plane)
    verts = [plane.matrix_world @ v.co for v in plane_mesh.vertices]
    
    if len(verts) < 4:
        raise ValueError("Target plane must have at least 4 vertices")
    
    # For a simple rectangular plane, use first 4 verts
    v0 = verts[0]
    v1 = verts[1]
    v2 = verts[2]
    v3 = verts[3]
    
    # Create plane basis vectors
    edge1 = v1 - v0
    edge2 = v3 - v0
    
    # Project world point onto plane
    to_point = world_point - v0
    
    u = to_point.dot(edge1) / edge1.dot(edge1)
    v = to_point.dot(edge2) / edge2.dot(edge2)
    
    return (u, v)

In [6]:
from mathutils.geometry import barycentric_transform
class MaskProjector:
    def __init__(self, mask_path, camera_obj=None, output_path=None, texture_resolution=(2048, 2048)):
        self.mask_image = Image.open(mask_path).convert("RGBA")
        self.camera_obj = camera_obj or bpy.context.scene.camera
        self.output_path = output_path or os.path.join(os.path.dirname(mask_path), "projected_texture.png")
        self.in_texture_resolution = texture_resolution
        self.mask_width, self.mask_height = self.mask_image.size
        self.frame = self.camera_obj.data.view_frame(scene=bpy.context.scene)
        topRight = self.frame[0]
        bottomRight = self.frame[1]
        bottomLeft = self.frame[2]
        self.topLeft = self.frame[3]
        resolutionX = int(bpy.context.scene.render.resolution_x * (bpy.context.scene.render.resolution_percentage / 100))
        resolutionY = int(bpy.context.scene.render.resolution_y * (bpy.context.scene.render.resolution_percentage / 100))
        self.xRange = np.linspace(self.topLeft[0], topRight[0], resolutionX)
        self.yRange = np.linspace(self.topLeft[1], bottomLeft[1], resolutionY)
        self.output_texture = np.zeros((self.in_texture_resolution[1], self.in_texture_resolution[0], 4), dtype=np.uint8)

    def shoot_ray(self, xIndex: int, yIndex: int) -> Vector:
        origin = self.camera_obj.matrix_world.translation
        depsgraph = bpy.context.evaluated_depsgraph_get()
        x = self.xRange[xIndex]
        y = self.yRange[yIndex]
        pixelVector = Vector((x, y, self.topLeft[2]))
        destination = self.camera_obj.matrix_world @ pixelVector
        direction = (destination - origin).normalized()    
        result, location, normal, index, obj, matrix = bpy.context.scene.ray_cast(
        depsgraph,
        origin,
        direction
        )
        if result:
            return location,index,obj
        else:
            return None,None,None
    def get_uv_from_world_point(self, obj, world_point, face_index):
        """Get UV coordinates for a point on mesh surface."""
        # Get evaluated mesh (includes modifiers)
        depsgraph = bpy.context.evaluated_depsgraph_get()
        obj_eval = obj.evaluated_get(depsgraph)
        mesh = obj_eval.data
        
        bm = bmesh.new()
        bm.from_mesh(mesh)
        
        # Get UV layer
        uv_layer = bm.loops.layers.uv.active
        if uv_layer is None:
            raise ValueError("Mesh has no UV map")
        
        # Convert world point to local space of evaluated object
        local_point = obj_eval.matrix_world.inverted() @ world_point
        
        bm.faces.ensure_lookup_table()
        face = bm.faces[face_index]
        
        if len(face.verts) < 3:
            bm.free()
            return None
        
        # Use first 3 verts for triangle
        v0 = face.verts[0].co
        v1 = face.verts[1].co
        v2 = face.verts[2].co
        
        # Calculate barycentric coordinates
        bary = barycentric_transform(local_point, v0, v1, v2, Vector((1,0,0)), Vector((0,1,0)), Vector((0,0,1)))
        u, v, w = bary
        
        # Get UV coordinates from face loops
        uv0 = face.loops[0][uv_layer].uv
        uv1 = face.loops[1][uv_layer].uv
        uv2 = face.loops[2][uv_layer].uv
        
        # Interpolate UV using barycentric coordinates
        uv_result = u * uv0 + v * uv1 + w * uv2
        # print(uv0,uv1,uv2)
        # print(u, v, w)
        # print(local_point)
        # print(v0,v1,v2)
        bm.free()
        return uv_result

    def project_mask(self) -> np.ndarray:
        mask_array = np.array(self.mask_image)
        in_texture_resolution = self.in_texture_resolution
        # Get UV bounds of output texture
        output_texture = np.zeros((in_texture_resolution[1], in_texture_resolution[0], 4), dtype=np.uint8)
                # Process each pixel in mask
        for y in range(self.mask_height):
            for x in range(self.mask_width):
                alpha = mask_array[y, x, 3]
                        
                # Skip transparent pixels
                if alpha < 1:
                    continue
                        
                    # Shoot ray and get intersection
                intersection_point,face_index,obj = self.shoot_ray(x, y)
                        
                if intersection_point is None:
                    print(f"No intersection for pixel ({x}, {y})")
                    continue
                    
                        # Convert world point to plane UV
                        
                u, v = self.get_uv_from_world_point(obj, intersection_point, face_index)
                        
                        # Clamp to [0, 1]
                u = max(0, min(1, u))
                v = max(0, min(1, v))
                        
                # Convert to texture coordinates
                tex_x = int(u * (in_texture_resolution[0] - 1))
                tex_y = int(v * (in_texture_resolution[1] - 1))
                        
                # Clamp to texture bounds
                tex_x = max(0, min(in_texture_resolution[0] - 1, tex_x))
                tex_y = max(0, min(in_texture_resolution[1] - 1, tex_y))
                        
                # Copy pixel from mask to output texture, which has same resolution of input map
                self.output_texture[tex_y, tex_x] = mask_array[y, x]
        
        
        output_img = Image.fromarray(self.output_texture, mode='RGBA')
        output_img.save(self.output_path)

        return self.output_texture

cam = bpy.data.objects['Camera']
mp = MaskProjector(mask_path=os.path.join(ROOT_PATH, "mask02.png"),camera_obj=cam, output_path="output_mask.png")
output_mask = mp.project_mask()


No intersection for pixel (198, 34)
No intersection for pixel (199, 34)
No intersection for pixel (200, 34)
No intersection for pixel (201, 34)
No intersection for pixel (202, 34)
No intersection for pixel (203, 34)
No intersection for pixel (204, 34)
No intersection for pixel (205, 34)
No intersection for pixel (182, 35)
No intersection for pixel (183, 35)
No intersection for pixel (184, 35)
No intersection for pixel (185, 35)
No intersection for pixel (186, 35)
No intersection for pixel (187, 35)
No intersection for pixel (188, 35)
